# Combined Transcriptomics Cleaning
Processes 2019_UGA, 2020_UGA, and SDY2867 with a shared pipeline, then concatenates into a single parquet.

In [1]:
import pandas as pd

from data_cleaning.utils import log_standard_scale, peek

DATA_PATH = "../../data"
CLEAN_DATA_PATH = "../../cleaned_data"

In [2]:
unique_genes_challenge = set(
    pd.read_csv(DATA_PATH + '/challenge_transcriptomics.tsv', sep='\t')['ensembl_gene_id'].unique()
)
print(f"Challenge genes: {len(unique_genes_challenge)}")

Challenge genes: 54902


In [3]:
def clean_transcriptomics(filename: str) -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH + '/' + filename, sep='\t')
    df = df.drop(columns=['transcriptomics_id', 'raw_count', 'material'])

    # df = df[df['timepoint'].isin([0, 7])]
    df = df[df['ensembl_gene_id'].isin(unique_genes_challenge)]

    print(f"\n{filename} — timepoint value counts:")
    print(df['timepoint'].value_counts().sort_index())
    print(df.groupby('timepoint')['tpm_count'].mean())

    # df_pivot = df.pivot_table(
    #     index='participant_id',
    #     columns=['timepoint', 'ensembl_gene_id'],
    #     values='tpm_count'
    # )
    # df_pivot.columns = [f'TRAN_{gene}_d{int(tp)}' for tp, gene in df_pivot.columns]
    # df_pivot = df_pivot.reset_index()
    #
    # df_pivot = log_standard_scale(df_pivot)
    # return df_pivot
    return df

## Process each dataset

In [4]:
df_2019 = clean_transcriptomics('train_transcriptomics_2019_UGA.tsv')
df_2019.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_2019_UGA_cleaned.parquet', index=False)
print(f"2019_UGA: {df_2019.shape}")
peek(df_2019)


train_transcriptomics_2019_UGA.tsv — timepoint value counts:
timepoint
0     8586050
3     5363026
7     5363026
28    5363026
Name: count, dtype: int64
timepoint
0      30.417325
3     308.557208
7     276.796504
28    244.338500
Name: tpm_count, dtype: float64
2019_UGA: (24675128, 4)


,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2019_UGA.ID_208,0,ENSG00000000003,0.077926
1,2019_UGA.ID_208,0,ENSG00000000005,0.000000
2,2019_UGA.ID_208,0,ENSG00000000419,5.342548
3,2019_UGA.ID_208,0,ENSG00000000457,9.799336
4,2019_UGA.ID_208,0,ENSG00000000460,2.118530


In [5]:
df_2020 = clean_transcriptomics('train_transcriptomics_2020_UGA.tsv')
df_2020.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_2020_UGA_cleaned.parquet', index=False)
print(f"2020_UGA: {df_2020.shape}")
peek(df_2020)


train_transcriptomics_2020_UGA.tsv — timepoint value counts:
timepoint
0     1705200
28    1705200
Name: count, dtype: int64
timepoint
0     22.815432
28    22.459088
Name: tpm_count, dtype: float64
2020_UGA: (3410400, 4)


,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2020_UGA.ID_001,0,ENSG00000000003,0.607222
1,2020_UGA.ID_001,0,ENSG00000000005,0.000000
2,2020_UGA.ID_001,0,ENSG00000000419,68.417030
3,2020_UGA.ID_001,0,ENSG00000000457,5.885515
4,2020_UGA.ID_001,0,ENSG00000000460,2.958027


In [6]:
df_sdy = clean_transcriptomics('train_transcriptomics_SDY2867.tsv')
df_sdy.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_SDY2867_cleaned.parquet', index=False)
print(f"SDY2867: {df_sdy.shape}")
peek(df_sdy)


train_transcriptomics_SDY2867.tsv — timepoint value counts:
timepoint
-14    4007846
 0     3952944
 1     3952944
 7     3952944
 28    3952944
Name: count, dtype: int64
timepoint
-14    18.214273
 0     18.214273
 1     18.214273
 7     18.214273
 28    18.214273
Name: tpm_count, dtype: float64
SDY2867: (19819622, 4)


,participant_id,timepoint,ensembl_gene_id,tpm_count
0,SDY2867.SUB389725,0,ENSG00000000003,0.038336
1,SDY2867.SUB389727,-14,ENSG00000000003,0.200864
2,SDY2867.SUB389692,1,ENSG00000000003,0.964728
3,SDY2867.SUB389670,28,ENSG00000000003,0.670581
4,SDY2867.SUB389681,0,ENSG00000000003,0.094223


## Combine and save

In [7]:
df_combined = pd.concat([df_2019, df_2020, df_sdy], axis=0).reset_index(drop=True)
df_combined.to_parquet(CLEAN_DATA_PATH + '/transcriptomics_combined_cleaned.parquet', index=False)
print(f"Combined: {df_combined.shape}")
peek(df_combined)

Combined: (47905150, 4)


,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2019_UGA.ID_208,0,ENSG00000000003,0.077926
1,2019_UGA.ID_208,0,ENSG00000000005,0.000000
2,2019_UGA.ID_208,0,ENSG00000000419,5.342548
3,2019_UGA.ID_208,0,ENSG00000000457,9.799336
4,2019_UGA.ID_208,0,ENSG00000000460,2.118530
